# IAT-Based Fuzzy Logic Anomaly Detection for IoT Networks (CICIoT2023)
## Research Demonstration: Zero-Day & Unseen Attack Detection

This notebook demonstrates an interpretable, unsupervised network anomaly detection pipeline using **Inter-Arrival Time (IAT)** statistics and a **Mamdani Fuzzy Inference System** on the **CICIoT2023** dataset.

### Core Pipeline Architecture:
1. **CICIoT2023 Ingestion & Preprocessing** (Using existing `IAT` feature; no fabricated timestamps)
2. **Ordered Consecutive Observation Sliding Windows** ($W=20, \text{overlap}=50\%$)
3. **Statistical Feature Engineering** (Mean IAT, Std IAT, Coefficient of Variation $CV = \sigma/\mu$, Mean Packet Rate, IAT Deviation)
4. **Uncontaminated Normal Baseline** (Constructed strictly from normal `BenignTraffic`)
5. **Adaptive Median + MAD Thresholding** ($T_t = \text{Median} + k \cdot \text{MAD}$ with anti-poisoning protection)
6. **Mamdani Fuzzy Inference System** (Linguistic terms LOW, MEDIUM, HIGH, 12 interpretable rules, centroid defuzzification)
7. **Continuous Attack Risk Score** $\in [0, 1] \implies$ Normal, Suspicious, Attack
8. **Strict Zero-Day / Unseen Attack Evaluation** (Held-out attack family evaluation: Mirai Botnet)

In [ ]:
import sys
from pathlib import Path

# Ensure project root in python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import config
from src.data_loader import CICIoTDataLoader
from src.preprocessing import DataPreprocessor
from src.feature_engineering import WindowFeatureExtractor
from src.baseline import NormalBaselineBuilder, AdaptiveThresholdDetector
from src.fuzzy_system import IoTFuzzyAnomalyDetector
from src.zero_day_experiment import ZeroDayExperiment
from src.evaluation import PerformanceEvaluator

print("All modules loaded successfully!")

## 1. Load Data & Establish Normal Baseline
We extract pure `BenignTraffic` from the training partition to calculate baseline statistics without any attack contamination.

In [ ]:
loader = CICIoTDataLoader(config)
preprocessor = DataPreprocessor(config)

# Extract pure normal training data
raw_normal_train = loader.get_normal_training_data(max_samples=30000)
clean_normal_train = preprocessor.clean_and_validate(raw_normal_train, dataset_name="train_normal")

# Compute normal baseline windows and parameters
feature_extractor = WindowFeatureExtractor(cfg=config)
normal_windows = feature_extractor.extract_features(clean_normal_train)

baseline_builder = NormalBaselineBuilder(config)
baseline_stats = baseline_builder.fit(normal_windows)
feature_extractor.set_normal_baseline(baseline_stats["normal_mean_iat"])

print(f"Normal CV Median: {baseline_stats['normal_cv_median']:.4f}, MAD: {baseline_stats['normal_cv_mad']:.4f}")
print(f"Normal Rate Median: {baseline_stats['rate_percentiles']['p50']:.2f} pkts/s")

## 2. Load Multi-Phase Continuous Evaluation Scenario
We assemble a realistic evaluation scenario with:
- Normal operations
- Known attack bursts
- Recovery phases
- Novel Zero-Day attack bursts (Mirai)

In [ ]:
eval_stream = loader.load_evaluation_stream(total_records=25000)
clean_eval = preprocessor.clean_and_validate(eval_stream, dataset_name="eval_scenario")
eval_windows = feature_extractor.extract_features(clean_eval)
print(f"Extracted {len(eval_windows):,} sliding windows for evaluation.")

## 3. Initialize & Execute Mamdani Fuzzy Inference System
The fuzzy system uses triangular and trapezoidal membership functions and 12 interpretable domain rules.

In [ ]:
fuzzy_detector = IoTFuzzyAnomalyDetector(baseline_stats=baseline_stats, cfg=config)
risk_scores, categories, fuzzy_preds = fuzzy_detector.predict_batch(eval_windows)

eval_windows["attack_risk"] = risk_scores
eval_windows["fuzzy_category"] = categories
eval_windows["fuzzy_pred"] = fuzzy_preds

print(f"Mean Attack Risk across test windows: {np.mean(risk_scores):.4f}")
print("Fuzzy Category breakdown:")
print(eval_windows["fuzzy_category"].value_counts())

## 4. Run Baseline Methods (Simple CV & Adaptive Median + MAD)

In [ ]:
simple_detector = AdaptiveThresholdDetector(baseline_stats=baseline_stats, cfg=config)
adaptive_detector = AdaptiveThresholdDetector(baseline_stats=baseline_stats, cfg=config)

pred_simple, _ = simple_detector.predict_simple_cv(eval_windows)
res_adaptive = adaptive_detector.run_adaptive_detection(eval_windows, confidence_risk_scores=risk_scores)

eval_windows["simple_pred"] = pred_simple
eval_windows["adaptive_pred"] = res_adaptive["predictions"]
eval_windows["adaptive_threshold"] = res_adaptive["adaptive_thresholds"]

## 5. Performance Comparison
Compare Accuracy, Precision, Recall, F1-Score, and Zero-Day Detection Rate across all three methods.

In [ ]:
evaluator = PerformanceEvaluator(config)
y_true = eval_windows["binary_label"].to_numpy()

comparison_df = evaluator.compare_methods(
    y_true=y_true,
    preds_dict={
        "Simple CV": eval_windows["simple_pred"].to_numpy(),
        "Adaptive MAD": eval_windows["adaptive_pred"].to_numpy(),
        "Fuzzy Logic": eval_windows["fuzzy_pred"].to_numpy(),
    }
)
display(comparison_df[["method", "accuracy", "precision", "recall_tpr", "f1_score", "false_positive_rate"]])

## 6. Visualizing Fuzzy Attack Risk Timeline
Notice how the Attack Risk transitions cleanly between Normal (< 0.35), Suspicious (0.35 - 0.65), and Attack (>= 0.65).

In [ ]:
plt.figure(figsize=(14, 5))
n_pts = min(400, len(eval_windows))
sub_w = eval_windows.iloc[:n_pts]

plt.axhspan(0.0, 0.35, color='#d4edda', alpha=0.5, label='Normal Zone')
plt.axhspan(0.35, 0.65, color='#fff3cd', alpha=0.5, label='Suspicious Zone')
plt.axhspan(0.65, 1.0, color='#f8d7da', alpha=0.5, label='Attack Zone')

plt.plot(sub_w["window_id"], sub_w["attack_risk"], color='#343a40', lw=1.8, label='Attack Risk Score')

att = sub_w[sub_w["binary_label"] == 1]
plt.scatter(att["window_id"], att["attack_risk"], color='#dc3545', s=20, label='Ground Truth Attack')

plt.title("Mamdani Fuzzy Attack Risk Score over Sliding Windows", fontsize=13)
plt.xlabel("Window ID")
plt.ylabel("Attack Risk [0, 1]")
plt.ylim(-0.05, 1.05)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()